# Week 01 · Day 04 — LLM Evaluation

**Topics:** RAGAS faithfulness · Answer relevancy · LLM-as-judge · Evaluation pipeline


In [ ]:
%pip install -q openai ragas datasets chromadb langchain-openai

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1 · Why Evaluate?

A RAG pipeline that "seems to work" on 5 manual tests is not production-ready. Structured evaluation gives you:
- A number to track over time (did the last prompt change improve things?)
- Coverage across failure modes (hallucination, irrelevance, incomplete retrieval)
- A gate before deployment

In [ ]:
# Build a small RAG pipeline to evaluate
import chromadb

chroma = chromadb.Client()
col = chroma.get_or_create_collection("eval_demo")

knowledge_base = [
    "Python was created by Guido van Rossum and first released in 1991.",
    "Python's name comes from Monty Python's Flying Circus, not the snake.",
    "Python is dynamically typed and uses indentation to define code blocks.",
    "The Python Package Index (PyPI) hosts over 400,000 packages as of 2024.",
    "Python 3.0 was released in 2008 and is not backward compatible with Python 2.",
    "GIL (Global Interpreter Lock) prevents true multi-threaded parallelism in CPython.",
    "asyncio enables asynchronous programming in Python using async/await syntax.",
    "List comprehensions are a Pythonic way to create lists: [x**2 for x in range(10)].",
]

def embed(texts):
    r = client.embeddings.create(input=texts, model="text-embedding-3-small")
    return [item.embedding for item in r.data]

col.add(
    documents=knowledge_base,
    embeddings=embed(knowledge_base),
    ids=[f"doc_{i}" for i in range(len(knowledge_base))],
)

def rag(question: str, n: int = 3) -> tuple[str, list[str]]:
    q_emb = embed([question])[0]
    results = col.query(query_embeddings=[q_emb], n_results=n)
    contexts = results["documents"][0]
    context_str = "\n".join(contexts)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer using only the provided context. Be concise."},
            {"role": "user", "content": f"Context: {context_str}\n\nQuestion: {question}"},
        ],
        temperature=0,
        max_tokens=200,
    )
    return response.choices[0].message.content, contexts

print("RAG pipeline ready")

## 2 · RAGAS Evaluation

RAGAS metrics:
- **Faithfulness**: is every claim in the answer supported by the retrieved context?
- **Answer relevancy**: how directly does the answer address the question?
- **Context recall**: does the retrieved context contain all the information needed to answer? (requires ground truth)
- **Context precision**: how relevant are the retrieved chunks? (requires ground truth)

In [ ]:
# Build a test set
test_questions = [
    "Who created Python?",
    "Where does the name Python come from?",
    "What is the GIL?",
    "How does asyncio work in Python?",
    "When was Python 3.0 released?",
]

ground_truths = [
    "Python was created by Guido van Rossum.",
    "Python's name comes from Monty Python's Flying Circus.",
    "The GIL (Global Interpreter Lock) prevents true multi-threaded parallelism in CPython.",
    "asyncio enables asynchronous programming in Python using async/await syntax.",
    "Python 3.0 was released in 2008.",
]

# Generate answers and collect contexts
answers = []
all_contexts = []

for q in test_questions:
    answer, contexts = rag(q)
    answers.append(answer)
    all_contexts.append(contexts)
    print(f"Q: {q}")
    print(f"A: {answer[:100]}\n")

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision

test_dataset = Dataset.from_dict({
    "question": test_questions,
    "answer": answers,
    "contexts": all_contexts,
    "ground_truth": ground_truths,
})

results = evaluate(
    test_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision],
)

print(results)

In [ ]:
# View per-question breakdown
df = results.to_pandas()
print(df[["question", "faithfulness", "answer_relevancy"]].to_string())

## 3 · LLM-as-Judge (No Ground Truth Needed)

When you don't have labeled ground truth, use a capable LLM to score responses. Good for production monitoring.

In [ ]:
def llm_judge(question: str, answer: str, context: str) -> dict:
    prompt = f"""Evaluate this RAG response on two criteria. Respond in JSON.

Question: {question}
Context: {context}
Answer: {answer}

Score each criterion from 0.0 to 1.0:
- faithfulness: Is every claim in the answer supported by the context? (1.0 = fully grounded, 0.0 = hallucinated)
- relevance: Does the answer directly address the question? (1.0 = fully relevant, 0.0 = irrelevant)

Respond with JSON: {{"faithfulness": 0.0-1.0, "relevance": 0.0-1.0, "reason": "brief explanation"}}"""

    import json
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(r.choices[0].message.content)

# Evaluate a sample
q = test_questions[0]
a, ctx = rag(q)
score = llm_judge(q, a, "\n".join(ctx))

print(f"Question: {q}")
print(f"Answer:   {a}")
print(f"Scores:   {score}")

In [ ]:
# Run LLM-as-judge on all test cases
judge_results = []
for q, a, ctx in zip(test_questions, answers, all_contexts):
    score = llm_judge(q, a, "\n".join(ctx))
    judge_results.append(score)

avg_faith = sum(r["faithfulness"] for r in judge_results) / len(judge_results)
avg_rel = sum(r["relevance"] for r in judge_results) / len(judge_results)

print(f"Average faithfulness: {avg_faith:.3f}")
print(f"Average relevance:    {avg_rel:.3f}")
print()
print("Gate check (fail if faithfulness < 0.80):")
print("PASS" if avg_faith >= 0.80 else "FAIL — do not deploy")

## 4 · Evaluation-Driven Improvement

Use evaluation scores to guide improvements. Common levers:
- Low faithfulness → strengthen the system prompt constraint
- Low context recall → increase `n_results`, use hybrid search, or HyDE
- Low answer relevancy → improve the prompt or add query rewriting

In [ ]:
def rag_strict(question: str, n: int = 3) -> tuple[str, list[str]]:
    """Stricter system prompt to improve faithfulness."""
    q_emb = embed([question])[0]
    results = col.query(query_embeddings=[q_emb], n_results=n)
    contexts = results["documents"][0]
    context_str = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer ONLY using the provided context. "
                    "Do NOT add any information not present in the context. "
                    "If the answer is not in the context, respond: 'I cannot answer from the provided context.' "
                    "Cite sources as [1], [2] etc."
                ),
            },
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {question}"},
        ],
        temperature=0,
        max_tokens=200,
    )
    return response.choices[0].message.content, contexts

# Compare strict vs original on a hallucination-prone question
q = "What programming languages is Guido van Rossum known for besides Python?"
original_answer, _ = rag(q)
strict_answer, _ = rag_strict(q)

print(f"Original: {original_answer}")
print()
print(f"Strict:   {strict_answer}")

## Summary

- **RAGAS**: uses LLMs internally to score faithfulness, relevancy, recall, precision. Run offline before deployment.
- **LLM-as-judge**: flexible, no ground truth needed, but costs tokens. Use for production sampling.
- **Deployment gate**: set minimum thresholds (faithfulness ≥ 0.80) and block release if not met.
- Low scores tell you *where* the pipeline is failing — fix retrieval, chunking, or the prompt.

**Next:** [Responsible AI](week01-day04-responsible-ai.ipynb)